In [37]:
import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from ax.utils.stats.model_fit_stats import MSE
from botorch.models import SingleTaskGP
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import plotly.express as px

from gpytorch.kernels import MaternKernel
from ax.models.torch.botorch_modular.kernels import DefaultRBFKernel, ScaleMaternKernel
from gpytorch.kernels.linear_kernel import LinearKernel
from gpytorch.kernels.rbf_kernel import RBFKernel

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Build Dataset

In [38]:
GrdSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A1_PGCI-GrdSrch-[27]-P3O1/raw-data_2023-03-10_PtA1-PGCI-GrdSrch-[27]-P3O1_Stykke-4.csv")
RndSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A2_PGCI-RndSrch-[27]-P3O1/raw-data_2023-03-10_PtA2-PGCI-RndSrch-[27]-P3O1_Stykke-4.csv")
GrdSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
RndSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_8SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B5_PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1/raw-data_2023-03-20_PtB5-PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1-Stykke-4.csv")
BOpt_8SP_3It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)

In [39]:
# GrdSrch_Short = GrdSrch_df.drop(index=[0,2,4,6,8,10,12,14,16,18,20,22,24])

In [ ]:
# df = pd.concat(objs=[GrdSrch_df,RndSrch_df,BOpt_8SP_3It_df]) # Model mk16 using nu = 0.5 and obtaining RMSE = 1.148
# df = pd.concat(objs=[GrdSrch_df,RndSrch_df,BOpt_8SP_3It_df]) # Model mk17 using nu = 1.5 and obtaining RMSE = 1.142
# df = pd.concat(objs=[GrdSrch_df,RndSrch_df,BOpt_8SP_3It_df]) # Model mk 18 using nu = 2.5 and obtaining RMSE = 1.144
df.drop(columns=["mould_position","G_stoichiometry","CA_stoichiometry","IA_stoichiometry","StartPolymerMass_g","EndPolymerMass_pct"],inplace=True)
df['DeltaPolymerMass_pct']=df['DeltaPolymerMass_pct']*-1
df

,s1,s2,b1,DeltaPolymerMass_pct
0,0.8000,1.0000,0.8000,13.360997
1,1.0000,0.4000,0.6000,12.188133
2,0.4000,0.6000,0.8000,14.405027
3,0.8000,0.8000,1.0000,10.823033
4,0.4000,0.8000,0.6000,14.041169
...,...,...,...,...
24,0.4413,0.9805,0.5755,11.473868
25,0.4925,0.8819,0.5811,12.517877
26,0.3829,0.6579,0.7756,16.666833
27,0.3349,0.6460,0.7712,17.279822


In [41]:
X = df[["s1","s2","b1"]].to_numpy()
y = df["DeltaPolymerMass_pct"].to_numpy()

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Data Visualisation

In [43]:
fig = px.scatter_3d(df, x='s1', y='s2', z='b1', color='DeltaPolymerMass_pct',width=1300, height=600)
fig.show()

In [44]:
d = {"s1": X_train[:,0], "s2": X_train[:,1], "b1": X_train[:,2],"DeltaPolymerMass_pct": y_train}
train_df = pd.DataFrame(data=d)

fig = px.scatter_3d(train_df, x='s1', y='s2', z='b1', color='DeltaPolymerMass_pct',width=1300, height=600)
fig.show()

In [45]:
d = {"s1": X_test[:,0], "s2": X_test[:,1], "b1": X_test[:,2],"DeltaPolymerMass_pct": y_test}
test_df = pd.DataFrame(data=d)

fig = px.scatter_3d(test_df, x='s1', y='s2', z='b1', color='DeltaPolymerMass_pct',width=1300, height=600)
fig.show()

# Train Model

In [46]:
client = Client()

parameters = [
    RangeParameterConfig(
        name="s1", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="s2", parameter_type="float", bounds=(0, 1)
    ),
    RangeParameterConfig(
        name="b1", parameter_type="float", bounds=(0, 1)
    ),
]

client.configure_experiment(parameters=parameters)

In [47]:
def construct_generation_strategy(
    generator_spec: GeneratorSpec, node_name: str,
) -> GenerationStrategy:
    """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
    using the provided `generator_spec` for the Modular BoTorch node.
    """
    botorch_node = GenerationNode(
        node_name=node_name,
        model_specs=[generator_spec],
    )
    return GenerationStrategy(
        name=f"{node_name}",
        nodes=[botorch_node]
    )

construct_generation_strategy(
    generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
    node_name="Modular BoTorch",
)

GenerationStrategy(name='Modular BoTorch', nodes=[GenerationNode(node_name='Modular BoTorch', model_specs=[GeneratorSpec(model_enum=BoTorch, model_key_override=None)], transition_criteria=[])])

In [48]:
surrogate_spec = SurrogateSpec(
    model_configs=[
        ModelConfig(
            botorch_model_class=SingleTaskGP,

            # covar_module_class=MaternKernel,
            # covar_module_options={"nu": 2.5},

            covar_module_class=MaternKernel,
            covar_module_options={"nu": 1.5},

            # covar_module_class=MaternKernel,
            # covar_module_options={"nu": 0.5},

            # covar_module_class=RBFKernel,
        ),
    ],
    eval_criterion=MSE,
    allow_batched_models=False,
)

In [49]:
generator_spec = GeneratorSpec(
    model_enum=Generators.BOTORCH_MODULAR,
    model_kwargs={
        "surrogate_spec": surrogate_spec,
        "botorch_acqf_class": qLogNoisyExpectedImprovement,
        "acquisition_options": {},
    },
    model_gen_kwargs = {
        "model_gen_options": {
            "optimizer_kwargs": {
                "num_restarts": 20,
                "sequential": False,
                "options": {
                    "batch_limit": 5,
                    "maxiter": 200,
                },
            },
        },
    }
)

In [50]:
generation_strategy = construct_generation_strategy(
    generator_spec=generator_spec,
    node_name="BoTorch w/ Model Selection",
)
client.set_generation_strategy(
    generation_strategy=generation_strategy,
)

In [51]:
metric_name = "t1"
objective = f"{metric_name}"

client.configure_optimization(objective=objective)

In [52]:
for array,target in zip(X_train,y_train):
    my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
    trial_index = client.attach_trial(parameters=my_parameters)
    client.complete_trial(trial_index=trial_index,raw_data={"t1": target})

In [53]:
client.summarize()

,trial_index,arm_name,trial_status,t1,s1,s2,b1
0,0,0_0,COMPLETED,13.875438,0.200000,1.000000,0.600000
1,1,1_0,COMPLETED,13.685338,0.215714,0.153334,0.846572
2,2,2_0,COMPLETED,10.721822,0.080100,0.429500,0.147600
3,3,3_0,COMPLETED,13.211387,0.646300,0.461500,0.631200
4,4,4_0,COMPLETED,11.305263,0.200000,1.000000,0.200000
5,5,5_0,COMPLETED,20.775697,0.337000,0.647600,0.773100
6,6,6_0,COMPLETED,13.441874,0.574273,0.838241,0.726713
7,7,7_0,COMPLETED,11.542312,1.000000,0.400000,0.200000
8,8,8_0,COMPLETED,13.156327,0.111800,0.026700,0.216200
9,9,9_0,COMPLETED,11.598718,0.800000,0.600000,0.800000


# Visualising Model

In [54]:
client.get_next_trials(max_trials=1)
client.predict([{"s1":0.1,"s2":0.1,"b1":0.1}])[0]["t1"][0]

n = 10
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoordsOld_lis = []
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoordsOld_lis.append([i,j,k])
            ijkCoords_lis.append({"s1":i,"s2":j,"b1":k})
y_pred = client.predict(ijkCoords_lis)
y_pred_lis = []
for i in y_pred:
    y_pred_lis.append(i["t1"][0])
ijkCoordsOld_arr = np.array(ijkCoordsOld_lis)
y_pred_arr = np.array(y_pred_lis)
df2 = pd.DataFrame({'s1': ijkCoordsOld_arr[:, 0],'s2': ijkCoordsOld_arr[:, 1],'b1': ijkCoordsOld_arr[:, 2], 'y_pred': y_pred_arr})

fig = px.scatter_3d(df2, x='s1', y='s2', z='b1', color='y_pred')
fig.show()

print(np.max(y_pred_arr))
print(ijkCoordsOld_lis[np.argmax(y_pred_arr)])
print(df2)

17.433387108623716
[np.float64(0.25), np.float64(0.625), np.float64(0.75)]
      s1   s2     b1     y_pred
0    0.0  0.0  0.000  12.493045
1    0.0  0.0  0.125  12.737783
2    0.0  0.0  0.250  12.935907
3    0.0  0.0  0.375  12.979644
4    0.0  0.0  0.500  12.954056
..   ...  ...    ...        ...
724  1.0  1.0  0.500  11.707971
725  1.0  1.0  0.625  11.674872
726  1.0  1.0  0.750  11.641781
727  1.0  1.0  0.875  11.591689
728  1.0  1.0  1.000  11.564372

[729 rows x 4 columns]


# Test Model

In [56]:
y_pred = []
for i,j in zip(X_test,y_test):
    y_pred.append(client.predict([{"s1":i[0],"s2":i[1],"b1":i[2]}])[0]["t1"][0])
y_pred = np.array(y_pred)
root_mean_squared_error(y_test, y_pred)

1.1423314445642754

# Save Model

In [57]:
client.save_to_json_file('ModelMk17.json')